<a href="https://colab.research.google.com/github/lolndo/Basic_Networks1/blob/main/%D0%92%D0%B2%D0%B5%D0%B4%D0%B5%D0%BD%D0%B8%D0%B5_%D0%B2_%D0%BD%D0%B5%D0%B9%D1%80%D0%BE%D0%BD%D0%BD%D1%8B%D0%B5_%D1%81%D0%B5%D1%82%D0%B8_%D0%9B%D0%B8%D0%BD%D0%B5%D0%B9%D0%BD%D1%8B%D0%B9_%D1%81%D0%BB%D0%BE%D0%B9_(Dense)_%D0%94%D0%97_Lite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Задание

Создайте систему компьютерного зрения, которая будет определять тип геометрической фигуры. Используя подготовленную базу и шаблон ноутбука проведите серию экспериментов по перебору гиперпараметров нейронной сети, распознающей три категории изображений (треугольник, круг, квадрат).

1. Поменяйте количество нейронов в сети, используя следующие значения:

- один слой 10 нейронов
- один слой 100 нейронов
- один слой 5000 нейронов.

2. Поменяйте активационную функцию в скрытых слоях с `relu` на `linear`.
3. Поменяйте размеры batch_size:
- 10
- 100
- 1000

4. Выведите на экран получившиеся точности.

Всего должно получиться 18 комбинаций указанных параметров.

Создайте сравнительную таблицу по результатам проведенных тестов.

In [ ]:
# Подключение класса для создания нейронной сети прямого распространения
from tensorflow.keras.models import Sequential
# Подключение класса для создания полносвязного слоя
from tensorflow.keras.layers import Dense
# Подключение оптимизатора
from tensorflow.keras.optimizers import Adam
# Подключение утилит для to_categorical
from tensorflow.keras import utils
# Подключение библиотеки для загрузки изображений
from tensorflow.keras.preprocessing import image
# Подключение библиотеки для работы с массивами
import numpy as np
# Подключение библиотек для отрисовки изображений
import matplotlib.pyplot as plt
# Подключение модуля для работы с файлами
import os
# Вывод изображения в ноутбуке, а не в консоли или файле
%matplotlib inline

In [ ]:
# Загрузка датасета из облака
import gdown
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l3/hw_light.zip', None, quiet=True)

'hw_light.zip'

In [ ]:
# Распаковываем архив hw_light.zip в папку hw_light
!unzip -q hw_light.zip

In [ ]:
# Путь к директории с базой
base_dir = '/content/hw_light'
# Создание пустого списка для загрузки изображений обучающей выборки
x_train = []
# Создание списка для меток классов
y_train = []
# Задание высоты и ширины загружаемых изображений
img_height = 20
img_width = 20
# Перебор папок в директории базы
for patch in os.listdir(base_dir):
    # Перебор файлов в папках
    for img in os.listdir(base_dir + '/' + patch):
        # Добавление в список изображений текущей картинки
        x_train.append(image.img_to_array(image.load_img(base_dir + '/' + patch + '/' + img,
                                                    target_size=(img_height, img_width),
                                                    color_mode='grayscale')))
        # Добавление в массив меток, соответствующих классам
        if patch == '0':
            y_train.append(0)
        elif patch == '3':
            y_train.append(1)
        else:
            y_train.append(2)

# Преобразование в numpy-массив загруженных изображений и меток классов
x_train = np.array(x_train)
y_train = np.array(y_train)
# Вывод размерностей
print('Размер массива x_train', x_train.shape)
print('Размер массива y_train', y_train.shape)

Размер массива x_train (302, 20, 20, 1)
Размер массива y_train (302,)


In [ ]:
# Ячейка 1: Подготовка данных
from tensorflow.keras.layers import Flatten
from tensorflow.keras import utils
import numpy as np

# Нормализация изображений (приведение к диапазону [0, 1])
x_train = x_train.astype('float32') / 255.0

# Преобразование меток в one-hot encoding
y_train_cat = utils.to_categorical(y_train, 3)

# Фиксируем случайное зерно для воспроизводимости
import tensorflow as tf
tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
# Ячейка 2: Эксперименты с гиперпараметрами
neurons_list = [10, 100, 5000]
activations = ['relu', 'linear']
batch_sizes = [10, 100, 1000]
epochs = 20

results = []

for neurons in neurons_list:
    for act in activations:
        for batch in batch_sizes:
            print(f'Обучение: нейронов={neurons}, активация={act}, batch_size={batch}')

            # Создание модели
            model = Sequential([
                Flatten(input_shape=(20, 20, 1)),
                Dense(neurons, activation=act),
                Dense(3, activation='softmax')
            ])

            # Компиляция
            model.compile(optimizer=Adam(),
                          loss='categorical_crossentropy',
                          metrics=['accuracy'])

            # Обучение (без вывода прогресса для чистоты, verbose=0)
            history = model.fit(x_train, y_train_cat,
                                batch_size=batch,
                                epochs=epochs,
                                verbose=1)

            # Оценка точности на обучающей выборке
            accuracy = history.history['accuracy'][-1]
            print(f'Итоговая точность (train): {accuracy:.4f}\n')

            results.append({
                'Нейронов': neurons,
                'Активация': act,
                'Batch size': batch,
                'Точность': accuracy
            })

Обучение: нейронов=10, активация=relu, batch_size=10
Epoch 1/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4603 - loss: 1.0331
Epoch 2/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6589 - loss: 0.8433
Epoch 3/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7252 - loss: 0.7412
Epoch 4/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7285 - loss: 0.6867
Epoch 5/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7616 - loss: 0.6467
Epoch 6/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7914 - loss: 0.6145
Epoch 7/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8046 - loss: 0.5832
Epoch 8/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8179 - loss: 0.5484
Epoch 9/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8344 - loss: 0.5255
Epoch 10/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8377 - loss: 0.4972
Epoch 11/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8444 - loss: 0.4740
Epoch 12/20
31/31 ━━━

In [ ]:
import pandas as pd
df = pd.DataFrame(results)
df = df.sort_values(by=['Нейронов', 'Активация', 'Batch size']).reset_index(drop=True)
print(df.to_string(index=False))

 Нейронов Активация  Batch size  Точность
       10    linear          10  0.880795
       10    linear         100  0.741722
       10    linear        1000  0.559603
       10      relu          10  0.900662
       10      relu         100  0.572848
       10      relu        1000  0.652318
      100    linear          10  0.880795
      100    linear         100  0.834437
      100    linear        1000  0.715232
      100      relu          10  1.000000
      100      relu         100  0.907285
      100      relu        1000  0.798013
     5000    linear          10  0.814570
     5000    linear         100  0.870861
     5000    linear        1000  0.552980
     5000      relu          10  1.000000
     5000      relu         100  0.950331
     5000      relu        1000  0.748344
